# 03 - Feature Engineering & Time-based Split

- Tao credit_history_length tu earliest_cr_line
- WOE-transform cac bien duoc chon
- Time-based train/validation/test split theo issue_d

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from src.paths import DATA_RAW, DATA_INTERIM, DATA_PROCESSED, MODELS, REPORTS_FIGURES

## Feature engineering

In [2]:
import numpy as np
from src.features.clean import engineer_features, winsorize

accepted = pd.read_parquet(DATA_INTERIM / 'accepted_vintage_2015_2017.parquet')

with open(DATA_INTERIM / 'shortlist_features.txt') as f:
    SHORTLIST = [line.strip() for line in f if line.strip()]
print(f'Shortlist tu notebook 02 (IV > 0.02, {len(SHORTLIST)} bien):', SHORTLIST)

RAW_COLS = [
    'loan_amnt', 'emp_length', 'home_ownership', 'annual_inc', 'purpose', 'dti',
    'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high',
    'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
    'addr_state',
]
PRICING_COLS = ['int_rate', 'grade']
TARGET = 'loan_status'

df = accepted[['issue_d'] + RAW_COLS + PRICING_COLS + [TARGET]].copy()
df['bad_flag'] = (df[TARGET] == 'Charged Off').astype(int)
df = engineer_features(df)  # them emp_length_years, credit_history_length

print('\nShape:', df.shape)
df.head()


Shortlist tu notebook 02 (IV > 0.02, 8 bien): ['fico_range_low', 'fico_range_high', 'dti', 'annual_inc', 'home_ownership', 'inq_last_6mths', 'emp_length_years', 'revol_util']



Shape: (643917, 22)


,issue_d,loan_amnt,home_ownership,annual_inc,purpose,dti,delinq_2yrs,fico_range_low,fico_range_high,inq_last_6mths,...,revol_bal,revol_util,total_acc,addr_state,int_rate,grade,loan_status,bad_flag,emp_length_years,credit_history_length
0,2015-12-01,3600.0,MORTGAGE,55000.0,debt_consolidation,5.91,0.0,675.0,679.0,1.0,...,2765.0,29.7,13.0,PA,13.99,C,Fully Paid,0,10.0,148
1,2015-12-01,24700.0,MORTGAGE,65000.0,small_business,16.06,1.0,715.0,719.0,4.0,...,21470.0,19.2,38.0,SD,11.99,C,Fully Paid,0,10.0,192
2,2015-12-01,11950.0,RENT,34000.0,debt_consolidation,10.20,0.0,690.0,694.0,0.0,...,8822.0,68.4,6.0,GA,13.44,C,Fully Paid,0,4.0,338
3,2015-12-01,20000.0,MORTGAGE,180000.0,debt_consolidation,14.67,0.0,680.0,684.0,0.0,...,87329.0,84.5,27.0,MN,9.17,B,Fully Paid,0,10.0,306
4,2015-12-01,20000.0,MORTGAGE,85000.0,major_purchase,17.61,1.0,705.0,709.0,0.0,...,826.0,5.7,15.0,SC,8.49,B,Fully Paid,0,10.0,202


## Time-based split

In [3]:
from scipy.stats import chi2_contingency

df_sorted = df.sort_values('issue_d')
n = len(df_sorted)
train_cutoff = df_sorted['issue_d'].iloc[int(n * 0.7)]
val_cutoff = df_sorted['issue_d'].iloc[int(n * 0.85)]

print(f'Train: issue_d < {train_cutoff.date()}')
print(f'Val:   {train_cutoff.date()} <= issue_d < {val_cutoff.date()}')
print(f'Test:  issue_d >= {val_cutoff.date()}')

train_df = df[df['issue_d'] < train_cutoff].copy()
val_df = df[(df['issue_d'] >= train_cutoff) & (df['issue_d'] < val_cutoff)].copy()
test_df = df[df['issue_d'] >= val_cutoff].copy()

for name, part in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f'{name}: {len(part):,} ({len(part) / n:.1%}), bad rate {part.bad_flag.mean():.2%}')

# Kiem tra bad rate on dinh giua cac giai doan (phat hien vintage effect) - PROPOSAL muc 5.4
contingency = pd.DataFrame({
    'bad': [train_df.bad_flag.sum(), val_df.bad_flag.sum(), test_df.bad_flag.sum()],
    'good': [
        len(train_df) - train_df.bad_flag.sum(),
        len(val_df) - val_df.bad_flag.sum(),
        len(test_df) - test_df.bad_flag.sum(),
    ],
}, index=['train', 'val', 'test'])
chi2, p_value, _, _ = chi2_contingency(contingency)
print(f'\nChi-square bad rate train/val/test: chi2={chi2:.2f}, p={p_value:.4f}')
print('-> Bad rate KHAC BIET co y nghia thong ke giua cac giai doan (vintage effect).' if p_value < 0.05
      else '-> Bad rate on dinh giua cac giai doan, khong co bang chung vintage effect ro ret.')

# Winsorize: fit bounds CHI TREN train, ap dung lai cho val/test - tranh leakage nguong cat
train_df, winsor_bounds = winsorize(train_df)
val_df, _ = winsorize(val_df, winsor_bounds)
test_df, _ = winsorize(test_df, winsor_bounds)


Train: issue_d < 2016-08-01
Val:   2016-08-01 <= issue_d < 2017-03-01
Test:  issue_d >= 2017-03-01


Train: 439,698 (68.3%), bad rate 16.31%
Val: 99,015 (15.4%), bad rate 21.48%
Test: 105,204 (16.3%), bad rate 19.94%

Chi-square bad rate train/val/test: chi2=1916.23, p=0.0000
-> Bad rate KHAC BIET co y nghia thong ke giua cac giai doan (vintage effect).


## WOE transform

In [4]:
import pickle
from optbinning import BinningProcess

FEATURES = [c for c in SHORTLIST if c in train_df.columns]
numeric_features = [c for c in FEATURES if pd.api.types.is_numeric_dtype(train_df[c])]
categorical_features = [c for c in FEATURES if c not in numeric_features]

# Fit CHI TREN train - khong duoc fit tren toan bo du lieu, neu khong bin edges se "nhin thay"
# thong tin tu val/test (tuong lai), pha vo tinh chat time-based split.
binning_process = BinningProcess(
    variable_names=FEATURES,
    categorical_variables=categorical_features,
)
binning_process.fit(train_df[FEATURES], train_df['bad_flag'])


def add_woe(df_part: pd.DataFrame) -> pd.DataFrame:
    woe = binning_process.transform(df_part[FEATURES], metric='woe')
    woe.columns = [c + '_woe' for c in woe.columns]
    return pd.concat([df_part.reset_index(drop=True), woe.reset_index(drop=True)], axis=1)


train_df = add_woe(train_df)
val_df = add_woe(val_df)
test_df = add_woe(test_df)

MODELS.mkdir(parents=True, exist_ok=True)
with open(MODELS / 'binning_process.pkl', 'wb') as f:
    pickle.dump(binning_process, f)

print('WOE features:', [c + '_woe' for c in FEATURES])
train_df.filter(like='_woe').describe().T


WOE features: ['fico_range_low_woe', 'fico_range_high_woe', 'dti_woe', 'annual_inc_woe', 'home_ownership_woe', 'inq_last_6mths_woe', 'emp_length_years_woe', 'revol_util_woe']


,count,mean,std,min,25%,50%,75%,max
fico_range_low_woe,439698.0,0.056541,0.422977,-0.399951,-0.268068,-0.070817,0.370129,1.099121
fico_range_high_woe,439698.0,0.056541,0.422977,-0.399951,-0.268068,-0.070817,0.370129,1.099121
dti_woe,439698.0,0.020982,0.248666,-0.527381,-0.147204,0.031726,0.265956,0.327645
annual_inc_woe,439698.0,0.020182,0.245142,-0.448780,-0.123438,0.000992,0.238043,0.461519
home_ownership_woe,439698.0,0.016108,0.218931,-0.215889,-0.215889,-0.057326,0.247112,0.247112
inq_last_6mths_woe,439698.0,0.015985,0.216316,-0.437336,-0.125133,0.173568,0.173568,0.173568
emp_length_years_woe,439698.0,0.040022,0.062012,-0.034072,-0.000065,0.006971,0.127054,0.127054
revol_util_woe,439698.0,0.005170,0.123847,-0.157697,-0.082299,-0.061551,0.084667,0.290505


## Lưu ra data/processed

In [5]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# loan_amnt giu lai du khong nam trong shortlist IV - can cho tinh Expected Net Return
# (gia/khoi luong khoan vay) o notebook 05, khong phai bien dau vao model.
KEEP_COLS = (
    ['issue_d', 'bad_flag', 'loan_status', 'loan_amnt'] + PRICING_COLS + FEATURES + [c + '_woe' for c in FEATURES]
)
KEEP_COLS = list(dict.fromkeys(KEEP_COLS))  # loai trung neu loan_amnt da nam trong FEATURES
train_df[KEEP_COLS].to_parquet(DATA_PROCESSED / 'train.parquet', index=False)
val_df[KEEP_COLS].to_parquet(DATA_PROCESSED / 'val.parquet', index=False)
test_df[KEEP_COLS].to_parquet(DATA_PROCESSED / 'test.parquet', index=False)

print('Da luu train/val/test vao data/processed/')
print({'train': train_df.shape, 'val': val_df.shape, 'test': test_df.shape})


Da luu train/val/test vao data/processed/
{'train': (439698, 30), 'val': (99015, 30), 'test': (105204, 30)}
